In [ ]:
import pandas as pd 
import numpy as np
import hashlib
import math
from csv import reader
import re
import os
import altair as alt
# import seaborn as sns
import matplotlib.pyplot as plt
# plt.rcParams['pdf.fonttype'] = 42
# plt.rcParams['ps.fonttype'] = 42
import re
pd.set_option('display.max_rows', None)
plt.style.use('seaborn-v0_8-dark-palette')


In [ ]:
def hash_str(to_hash): 
    stringToHash = to_hash.encode()
    return hashlib.sha256(stringToHash).hexdigest()

In [ ]:
def process_lbl(line_list, filename): 
    data_list = []
    i = 0
    name_line = ''
    
    if 'cfg' in filename: 
        setup = 'cfg'
    elif 'risc0' in filename: 
        setup = 'risc0'
    elif 'coral' in filename: 
        setup = 'coral'
    else: 
        setup = 'unknown' 

        
    if 'json' in filename: 
        app = 'json'
    elif 'toml' in filename:
        app = 'toml'
    elif filename[0]=='c': 
        app = 'c'
    else: 
        app = 'unknown'

    doc_name = (filename.split('/')[-1])[:-4]
        
    while (i < len(line_list)): 
        if i+1 < len(line_list) and '---------,---------,---------,---------' in line_list[i+1]:
            name_line = line_list[i].strip('\n')     
        elif "---------,---------,---------,---------" in line_list[i]: 
            pass
        else:   
            for line in reader([setup+','+app+','+name_line+','+line_list[i].strip()]):
                if setup=='coral':
                    to_append = [doc_name]+line[:2]+line[4:]
                else: 
                    to_append = [doc_name]+line[:2]+[line[4]]+[0,0]+[line[5]]+ line[5:] 
                if len(to_append)!= 13: 
                    print(to_append)
                data_list.append(to_append)
        i = i + 1
    return data_list        

In [ ]:
def process_mem(line_list, filename): 
    data_list = []
    i = 0
    name_line = ''
    setup = ''
    app = ''
    doc = ''
    regex = ''
    batch_size = 0

    if 'cfg' in filename: 
        setup = 'cfg'
    elif 'risc0' in filename: 
        setup = 'risc0'
    elif 'coral' in filename: 
        setup = 'coral'
    else: 
        setup = 'unknown' 

    while (i < len(line_list)): 
        if 'Command being timed:' in line_list[i]:
            name_line = line_list[i]

            if 'json' in filename: 
                app = 'json'
            elif 'toml' in filename:
                app = 'toml'
            elif 'c_' in filename:
                app = 'c'
            else: 
                app = 'unknown'

            doc_size = filename.split('_')[1]

            if setup == 'coral': 
                batch_size = re.findall("-b ([0-9]*)", name_line)[0]
            else: 
                bacth_size = doc_size

        elif "Maximum resident set size" in line_list[i]: 
            mem_usage = int(line_list[i].split(':')[1].strip())/1e3
            data_list.append([filename, app, setup, doc_size, batch_size, mem_usage])
        else: 
           pass
        i = i + 1
    return data_list     

In [ ]:
time_data = []
time_directory = 'tests/results/timings/scale'
 
# iterate over files in
# that directory
for filename in os.listdir(time_directory):
    filepath = os.path.join(time_directory, filename)
    if 'Data' not in filename and 'DS_Store' not in filename and os.path.isfile(filepath):
        print(filepath)
        f = open(filepath, "r")
        lbl = [x for x in f]
        f.close()
        time_data.extend(process_lbl(lbl,filename))

In [ ]:
time_df = pd.DataFrame(time_data, columns = ['doc_name','setup','app','time','n_rules','tree_size','doc_len','batch_size',
                                    'test_type','component','test','val','metric'])
num_cols = ['n_rules','tree_size','batch_size','val','doc_len']
time_df[num_cols] = time_df[num_cols].apply(pd.to_numeric,errors='coerce', axis=1)

In [ ]:
e2e = time_df.loc[time_df.test.str.contains('prove_e2e')].reset_index()
e2e_coral = time_df.loc[time_df.test.str.contains('prove_e2e')].groupby(['time','setup','app','doc_len','batch_size']).val.sum().reset_index()
df_e2e_avg = e2e_coral.groupby(['setup','app','batch_size','doc_len']).val.mean().reset_index()

e2e_avg_small = df_e2e_avg.iloc[df_e2e_avg.groupby('doc_len')["val"].nsmallest(1).index.get_level_values(1)].sort_values(by='doc_len')
e2e_nc = time_df.loc[(time_df.setup!='coral') & (time_df.test=='prove')]
df_e2e_nc_avg = e2e_nc.groupby(['setup','app','doc_len']).val.mean().reset_index().sort_values(by='doc_len')

In [ ]:
mem_data_coral = []
mem_directory = 'tests/results/memory/scale'
 
for filename in os.listdir(mem_directory):
    filepath = os.path.join(mem_directory, filename)
    if 'Data' not in filename and 'DS_Store' not in filename and os.path.isfile(filepath):
        print(filepath)
        f = open(filepath, "r")
        lbl = [x for x in f]
        f.close()
        mem_data_coral.extend(process_mem(lbl,filename))
        
# mem_df = pd.DataFrame(mem_data_coral)
mem_df = pd.DataFrame(mem_data_coral, columns = ['filename','app','setup','doc_len','batch_size','val'])
num_cols = ['val','doc_len','batch_size']
mem_df[num_cols] = mem_df[num_cols].apply(pd.to_numeric,errors='coerce', axis=1)

In [ ]:
mem_df_avg = mem_df.groupby(['filename','setup','app','doc_len','batch_size']).val.mean().reset_index()
mem_coral = mem_df_avg.loc[mem_df_avg.setup.str.contains("coral")]

In [ ]:
best_prove_time_combo = list(zip(e2e_avg_small.doc_len, e2e_avg_small.batch_size))
tuples_in_df = pd.MultiIndex.from_frame(mem_coral[["doc_len","batch_size"]])
mem_coral_small = mem_coral[tuples_in_df.isin(best_prove_time_combo)].sort_values(by=['doc_len'])

In [ ]:
mem_nc = mem_df_avg.loc[mem_df_avg.setup != "coral"].sort_values(by=['doc_len'])

In [ ]:
# Create a figure and an axes
# plt.rcParams.update({'font.size': 15})
fig, ax = plt.subplots()
ax.set_xscale('log', base = 2)
# ax.set_yscale('log')

# Group data by the 'line_id' column
# for line_name, group_data in df.groupby('line_id'):
#     Plot x_value vs y_value for each group
ax.plot(mem_coral_small.doc_len, mem_coral_small.val/1e3, marker='o')


i = 0; 
for line_name, group_data in mem_nc.groupby('setup'):
    if i == 0: 
        ax.plot(group_data.doc_len, group_data.val/1e3, linestyle='dashed', marker='X')
        i = i + 1
    else:
        ax.plot(group_data.doc_len, group_data.val/1e3, linestyle='dashed', marker='+')

# Add labels and title
ax.set_xlabel("Byte Stream Length", fontsize=16)
ax.set_ylabel("Maximum Memory Usage (GB)", fontsize=16)
ax.set_xticks([2**i for i in range(6,17)])
ax.set_facecolor('white')
# ax.set_title("Maximum Memory Usage")

# Add a legend to distinguish lines
# ax.legend()
ax.legend(['Coral', 'R0+Pest', 'R0+Custom'], loc=(.72,0), fontsize=12, frameon=False)
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)

# Display the plot
plt.savefig("compmem.pdf", format="pdf",bbox_inches='tight' )
plt.show()

In [ ]:
# Create a figure and an axes
fig, ax = plt.subplots()
# plt.rcParams.update({'font.size': 15})
ax.set_xscale('log', base = 2)
ax.set_yscale('log')

# Group data by the 'line_id' column
# for line_name, group_data in df.groupby('line_id'):
    # Plot x_value vs y_value for each group
ax.plot(e2e_avg_small.doc_len, e2e_avg_small.val/1e6, marker='o')

i = 0
for line_name, group_data in df_e2e_nc_avg.groupby('setup'):
    if i == 0: 
        m = 'X'
        i = i + 1
    else: 
        m = '+'
    ax.plot(group_data.doc_len, group_data.val/1e6, linestyle='dashed', marker=m)

ax.plot([2**i for i in range(6,17)],[5,5,6,9,15,40,90,175,400,900,2000], linestyle='dashdot',marker='v')
ax.plot([2**i for i in range(6,10)],[2,6.5,13,25],linestyle='dotted',marker="d")

# Add labels and title
ax.set_xlabel("Byte Stream Length",fontsize=16)
ax.set_ylabel("Total Prover time (s)",fontsize=16)
ax.set_xticks([2**i for i in range(6,17)])
ax.set_yticks([1,10,100])
plt.yticks([1,10,100,1000,10000], ['0','10','100','1000','10000'])
ax.set_facecolor('white')
# ax.set_title("Prover time as a function of Document Size")
plt.minorticks_off()

# Add a legend to distinguish lines
ax.legend(['Coral', 'R0+Pest', 'R0+Custom','MHNM VOLE*', 'MHNM SNARK*'], prop={'size': 10.5}, loc=(.66,.00), frameon=False)
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)
#Change to be shapes, and change colors, maybe fix log scale idk 
# Display the plot
plt.savefig("comptime.pdf", format="pdf",bbox_inches='tight')
plt.show()

In [ ]:
app_data = []
app_directory = 'tests/results/timings/apps'
 
# iterate over files in
# that directory
for filename in os.listdir(app_directory):
    filepath = os.path.join(app_directory, filename)
    if 'Data' not in filename and 'DS_Store' not in filename and os.path.isfile(filepath):
        print(filepath)
        f = open(filepath, "r")
        lbl = [x for x in f]
        f.close()
        app_data.extend(process_lbl(lbl,filename))

In [ ]:
mem_app_data_coral = []
mem_app_directory = 'tests/results/memory/apps'
 
for filename in os.listdir(mem_app_directory):
    filepath = os.path.join(mem_app_directory, filename)
    if 'Data' not in filename and 'DS_Store' not in filename and os.path.isfile(filepath):
        print(filepath)
        f = open(filepath, "r")
        lbl = [x for x in f]
        f.close()
        mem_app_data_coral.extend(process_mem(lbl,filename))
        
# mem_df = pd.DataFrame(mem_data_coral)
mem_app_df = pd.DataFrame(mem_app_data_coral, columns = ['filename','app','setup','doc_name','batch_size','val'])
num_cols = ['val','batch_size']
mem_app_df[num_cols] = mem_app_df[num_cols].apply(pd.to_numeric,errors='coerce', axis=1)

In [ ]:
mem_app_df_avg = mem_app_df.groupby(['filename','setup','app','doc_name','batch_size']).val.mean().reset_index()
mem_app_coral = mem_app_df_avg.loc[mem_app_df_avg.setup.str.contains("coral")]

In [ ]:
app_df = pd.DataFrame(app_data, columns = ['doc_name','setup','app','time','n_rules','tree_size','doc_len','batch_size',
                                    'test_type','component','test','val','metric'])
num_cols = ['n_rules','tree_size','batch_size','val','doc_len']
app_df[num_cols] = app_df[num_cols].apply(pd.to_numeric,errors='coerce', axis=1)

coral_app_df = app_df.loc[app_df.setup=='coral']


e2e_coral_app = coral_app_df.loc[coral_app_df.test.str.contains('prove_e2e')].groupby(['time',
                                                'app','doc_name','batch_size']).val.sum().reset_index()
df_e2e_avg_app = e2e_coral_app.groupby(['doc_name','app','batch_size']).val.mean().reset_index()
e2e_avg_small_app = df_e2e_avg_app.iloc[
    df_e2e_avg_app.groupby('doc_name')["val"].nsmallest(1).index.get_level_values(1)].sort_values(by='doc_name')

best_prove_time_combo_app = list(zip(e2e_avg_small_app.doc_name, e2e_avg_small_app.batch_size))
tuples_in_df_app = pd.MultiIndex.from_frame(coral_app_df[["doc_name","batch_size"]])
coral_df_app_filter = coral_app_df[tuples_in_df_app.isin(best_prove_time_combo_app)]

In [ ]:
cfg = app_df.loc[(app_df.setup=='cfg')&(app_df.test=='prove')]
risc0 = app_df.loc[(app_df.setup=='risc0')&(app_df.test=='prove')]
risc0_avg = risc0.groupby(['doc_name','app','test']).val.mean().reset_index()
cfg_avg = cfg.groupby(['doc_name','app','test']).val.mean().reset_index()

In [ ]:
tuples_in_df_app_mem = pd.MultiIndex.from_frame(mem_app_coral[["filename","batch_size"]])
to_latex = mem_app_coral[tuples_in_df_app_mem.isin(best_prove_time_combo_app)]
to_latex = to_latex.drop(columns='doc_name').rename(columns={"filename":'doc_name'})

In [97]:
cons = coral_df_app_filter.loc[coral_df_app_filter.metric=='constraints'].groupby(['doc_name']).val.mean().to_frame().reset_index()
to_latex = to_latex.merge(cons, on='doc_name',suffixes=['_mem','_cons'])
to_latex.val_mem = to_latex.val_mem/1e3

In [98]:
size = (coral_df_app_filter.loc[coral_df_app_filter.metric=='bytes'].groupby(['doc_name']).val.mean()/1e3).to_frame().reset_index()
to_latex = to_latex.merge(size, on='doc_name')

In [99]:
verify = (coral_df_app_filter.loc[(coral_df_app_filter.component=='V') & 
    (coral_df_app_filter.test.isin(["full_verify"]))].groupby(['doc_name',
                                                                    'time']).val.sum().reset_index().groupby(['doc_name']).val.mean()/1e6).to_frame().reset_index()
to_latex = to_latex.merge(verify, on='doc_name',suffixes=['_size','_verify'])

In [100]:
cg = coral_df_app_filter.loc[(
    (coral_df_app_filter.component=='G') | 
    (coral_df_app_filter.test=='witness_synthesis_0') |  
    (coral_df_app_filter.test.str.contains('nova_pp_gen')) | 
    (coral_df_app_filter.test.str.contains('sample_random_layer')) | 
    (coral_df_app_filter.test.str.contains('snark_params')) | 
    (coral_df_app_filter.test.str.contains('doc_commit'))
    )]
setup = (cg.groupby(['doc_name','test']).val.mean().reset_index().groupby(['doc_name']).val.sum()/1e6).to_frame().reset_index()
to_latex = to_latex.merge(setup, on='doc_name')

In [101]:
solve = (coral_df_app_filter.loc[coral_df_app_filter.test=='e2e_solving'].groupby(['doc_name']).val.mean()/1e6).to_frame().reset_index()
to_latex = to_latex.merge(solve, on='doc_name',suffixes=['_setup','_solve'])

In [ ]:
prove = (coral_df_app_filter.loc[coral_df_app_filter.test=='prove_e2e'].groupby(['doc_name']).val.mean()/1e6).to_frame().reset_index()
to_latex = to_latex.merge(prove, on='doc_name')

In [59]:
extra = coral_df_app_filter[['doc_name','doc_len','tree_size']].drop_duplicates()
to_latex = to_latex.merge(extra, on='doc_name')
to_latex.drop(columns='setup',inplace=True)

In [60]:
to_latex = to_latex[['app','doc_name','doc_len','tree_size','val_cons','batch_size','val_setup','val_solve','val',
                     'val_verify','val_size','val_mem']]
to_latex['doc_len'] = pd.to_numeric(to_latex['doc_len'], downcast='integer', errors='coerce')
to_latex['tree_size'] = pd.to_numeric(to_latex['tree_size'], downcast='integer', errors='coerce')
to_latex['batch_size'] = pd.to_numeric(to_latex['batch_size'], downcast='integer', errors='coerce')
to_latex['val_cons'] = pd.to_numeric(to_latex['val_cons'], downcast='integer', errors='coerce')


In [61]:
to_latex

,app,doc_name,doc_len,tree_size,val_cons,batch_size,val_setup,val_solve,val,val_verify,val_size,val_mem
0,c,c_c1_coral,56,241,89382,87,2.370425,0.119186,1.208434,0.031458,16.650,0.470983
1,c,c_c2_coral,237,1173,141229,155,2.872386,0.151462,2.154388,0.045854,17.194,0.764409
2,c,c_c3_coral,399,2027,211320,242,3.151509,0.171941,2.582602,0.041118,17.194,0.948934
3,c,c_llvm_coral,1431,6652,455274,543,5.193885,0.309483,5.920018,0.069689,18.282,2.385828
4,json,json_bank_citi_coral,2336,9503,413129,611,4.426142,0.397367,5.322595,0.054747,17.738,1.947452
5,json,json_bank_plaid_coral,312,1348,105668,141,2.464060,0.149746,1.876494,0.033501,16.650,0.525293
6,json,json_dk_coral,1547,7263,431620,641,4.496725,0.320780,4.400329,0.048782,17.738,1.984921
7,json,json_dmv_veratad_coral,748,2594,212606,305,3.116752,0.181047,2.457651,0.039674,17.194,0.970468
8,json,json_hibp_small_coral,191,869,102183,136,2.374245,0.134679,1.470004,0.031544,16.650,0.504752
9,json,json_jwt_coral,384,1777,145281,201,2.689020,0.155918,1.953409,0.037404,17.194,0.791569


In [58]:
print(to_latex.to_latex(index=False, float_format="{:,.3f}".format))

\begin{tabular}{llrrrrrrrrrr}
\toprule
app & doc_name & doc_len & tree_size & val_cons & batch_size & val_setup & val_solve & val & val_verify & val_size & val_mem \\
\midrule
c & c_c1_coral & 56 & 241 & 89382 & 87 & 2.370 & 0.119 & 1.208 & 0.031 & 16.650 & 0.471 \\
c & c_c2_coral & 237 & 1173 & 141229 & 155 & 2.872 & 0.151 & 2.154 & 0.046 & 17.194 & 0.764 \\
c & c_c3_coral & 399 & 2027 & 211320 & 242 & 3.152 & 0.172 & 2.583 & 0.041 & 17.194 & 0.949 \\
c & c_llvm_coral & 1431 & 6652 & 455274 & 543 & 5.194 & 0.309 & 5.920 & 0.070 & 18.282 & 2.386 \\
json & json_bank_citi_coral & 2336 & 9503 & 413129 & 611 & 4.426 & 0.397 & 5.323 & 0.055 & 17.738 & 1.947 \\
json & json_bank_plaid_coral & 312 & 1348 & 105668 & 141 & 2.464 & 0.150 & 1.876 & 0.034 & 16.650 & 0.525 \\
json & json_dk_coral & 1547 & 7263 & 431620 & 641 & 4.497 & 0.321 & 4.400 & 0.049 & 17.738 & 1.985 \\
json & json_dmv_veratad_coral & 748 & 2594 & 212606 & 305 & 3.117 & 0.181 & 2.458 & 0.040 & 17.194 & 0.970 \\
json & json_hib

In [59]:
def change_app_name_json(filename): 
    if "citi" in filename: 
        return "CitiBank"
    if "plaid" in filename: 
        return "Plaid"
    if "dk" in filename: 
        return "Sports"
    if "dmv" in filename: 
        return "Age"
    if "hibp_small" in filename: 
        return "HIBP"
    if "hibp" in filename: 
        return "X"
    if "jwt" in filename: 
        return "JWT"
    else: 
        return "Unknown"
        
def change_app_name(filename): 
    if "c1" in filename: 
        return "C1"
    if "c2" in filename: 
        return "C2"
    if "c3" in filename: 
        return "C3"
    if "llvm" in filename: 
        return "LLVM"
    if "toml_small" in filename: 
        return "T1"
    if "toml_coral" in filename: 
        return "T2"
    if "ark" in filename: 
        return "T3"
    else: 
        return "Unknown"
  
def change_test_name(test): 
    if test=="prove": 
        return "Baseline"
    if test=="Solving":
        return "Solving"
    if test=="prove_e2e":
        return "Prove"
    if test=="solving":
        return "Solving"
    else: 
        return "Unknown"

In [51]:
pt_coral = coral_df_app_filter.loc[coral_df_app_filter.test.str.contains('prove_e2e')].groupby([
                                                'app','doc_name','batch_size','test']).val.mean().reset_index()
pt_coral_solving_int = coral_df_app_filter.loc[coral_df_app_filter.test.str.contains('e2e_solving')].groupby([
                                                'app','doc_name','batch_size','time']).val.sum().reset_index()
pt_coral_solving = pt_coral_solving_int.groupby([
                                                'app','doc_name','batch_size']).val.mean().reset_index()
pt_coral_solving['test']='Solving'

pt_coral = pd.concat([pt_coral, pt_coral_solving])
json_coral = pt_coral.loc[pt_coral.app=='json']
json_cfg = cfg_avg.loc[cfg_avg.app=='json']
json_risc0 = risc0_avg.loc[risc0_avg.app=='json']

In [52]:
json_coral['Name'] = json_coral.doc_name.apply(lambda x: change_app_name_json(x))
json_cfg['Name'] = json_cfg.doc_name.apply(lambda x: change_app_name_json(x))
json_risc0['Name'] = json_risc0.doc_name.apply(lambda x: change_app_name_json(x))

json_coral['Component'] = json_coral.test.apply(lambda x: change_test_name(x))
json_cfg['Component'] = json_cfg.test.apply(lambda x: change_test_name(x))
json_risc0['Component'] = json_risc0.test.apply(lambda x: change_test_name(x))

/var/folders/y0/m5s5d3c1733_n9v6lmsxn3v00000gn/T/ipykernel_82723/2390281631.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  json_coral['Name'] = json_coral.doc_name.apply(lambda x: change_app_name_json(x))
/var/folders/y0/m5s5d3c1733_n9v6lmsxn3v00000gn/T/ipykernel_82723/2390281631.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  json_cfg['Name'] = json_cfg.doc_name.apply(lambda x: change_app_name_json(x))
/var/folders/y0/m5s5d3c1733_n9v6lmsxn3v00000gn/T/ipykernel_82723/2390281631.py:3: SettingWithCop

In [62]:
def prep_df(df, name):
    # df = df.stack().reset_index()
    df.columns = ['val', 'Name', 'Component']
    df.val = df.val/1e6
    df['Setup'] = name
    return df


In [63]:
df1 = prep_df(json_coral[['val', 'Name', 'Component']], 'Coral')
df2 = prep_df(json_cfg[['val', 'Name', 'Component']], 'R0+Pest')
df3 = prep_df(json_risc0[['val', 'Name', 'Component']], 'R0+Custom')

df = pd.concat([df1, df2, df3])

/var/folders/y0/m5s5d3c1733_n9v6lmsxn3v00000gn/T/ipykernel_75569/2225007191.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.val = df.val/1e6
/var/folders/y0/m5s5d3c1733_n9v6lmsxn3v00000gn/T/ipykernel_75569/2225007191.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Setup'] = name
/var/folders/y0/m5s5d3c1733_n9v6lmsxn3v00000gn/T/ipykernel_75569/2225007191.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = v

In [64]:
df = df.loc[df.Name!='X']
custom = {'Age':0, "CitiBank":1, "Plaid":2, "Sports":3, "HIBP":4, "JWT":5}
df = df.sort_values(by=['Name'],key=lambda x: x.map(custom))

In [66]:
 # chart = 
c1 = alt.Chart(df).mark_bar(clip=True).encode(

    # tell Altair which field to group columns on
    x=alt.X('Setup:N', title=None),

    # tell Altair which field to use as Y values and how to calculate
    y=alt.Y('sum(val):Q',
        axis=alt.Axis(
            grid=False,
            title="Total Prover Time")).scale(domain=(0,60)),

    # tell Altair which field to use to use as the set of columns to be  represented in each group
    column=alt.Column('Name:N', title=None,header=alt.Header(titleOrient='bottom', labelOrient='bottom',labelFontSize=15)).sort(['sadf']),
    order=alt.Order(
      # Sort the segments of the bars by this field
      'val',
      sort='descending'
    ),
    # tell Altair which field to use for color segmentation 
    color=alt.Color('Component:N',
            scale=alt.Scale(
                # make it look pretty with an enjoyable color pallet
                range=['#3d85c6', '#FF7F50','#b43709',"#d74814"],
            ),
        ))\
    .configure_view(
        # remove grid lines around column clusters
        strokeOpacity=0    
    ).configure_axis(
    labelFontSize=15,
    titleFontSize=15
).configure_legend(
     titleFontSize=15,
    labelFontSize=15,
    padding = 1,
    fillColor='white',
    orient = 'none', 
    legendX = 10,
    legendY = 40,
).properties(
    width=100,
    height=300
)
# c1.save("json_comp.pdf")

In [67]:
c1

alt.Chart(...)

In [65]:
# pt_coral = coral_df_filter.loc[coral_df_filter.test.str.contains('e2e|solving|sample')].groupby([
#                                                 'app','doc_name','batch_size','test']).val.mean().reset_index()
app_coral = pt_coral.loc[pt_coral.app!='json']
app_cfg = cfg_avg.loc[cfg_avg.app!='json']
app_risc0 = risc0_avg.loc[risc0_avg.app!='json']

In [66]:
app_coral['Name'] = app_coral.doc_name.apply(lambda x: change_app_name(x))
app_cfg['Name'] = app_cfg.doc_name.apply(lambda x: change_app_name(x))
app_risc0['Name'] = app_risc0.doc_name.apply(lambda x: change_app_name(x))

app_coral['Component'] = app_coral.test.apply(lambda x: change_test_name(x))
app_cfg['Component'] = app_cfg.test.apply(lambda x: change_test_name(x))
app_risc0['Component'] = app_risc0.test.apply(lambda x: change_test_name(x))

/var/folders/y0/m5s5d3c1733_n9v6lmsxn3v00000gn/T/ipykernel_63919/2585962810.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  app_coral['Name'] = app_coral.doc_name.apply(lambda x: change_app_name(x))
/var/folders/y0/m5s5d3c1733_n9v6lmsxn3v00000gn/T/ipykernel_63919/2585962810.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  app_cfg['Name'] = app_cfg.doc_name.apply(lambda x: change_app_name(x))
/var/folders/y0/m5s5d3c1733_n9v6lmsxn3v00000gn/T/ipykernel_63919/2585962810.py:3: SettingWithCopyWarning: 
A v

In [67]:
df1 = prep_df(app_coral[['val', 'Name', 'Component']], 'Coral')
df2 = prep_df(app_cfg[['val', 'Name', 'Component']], 'R0+Pest')
df3 = prep_df(app_risc0[['val', 'Name', 'Component']], 'R0+Custom')

df = pd.concat([df1, df2, df3])
df = df.sort_values(by='Name', ascending = True)

/var/folders/y0/m5s5d3c1733_n9v6lmsxn3v00000gn/T/ipykernel_63919/2225007191.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.val = df.val/1e6
/var/folders/y0/m5s5d3c1733_n9v6lmsxn3v00000gn/T/ipykernel_63919/2225007191.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Setup'] = name
/var/folders/y0/m5s5d3c1733_n9v6lmsxn3v00000gn/T/ipykernel_63919/2225007191.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = v

In [68]:
# chart = 
alt.Chart(df).mark_bar(clip=True).encode(

    # tell Altair which field to group columns on
    x=alt.X('Setup:N', title=None),

    # tell Altair which field to use as Y values and how to calculate
    y=alt.Y('sum(val):Q',
        axis=alt.Axis(
            grid=False,
            title="Total Prover Time")).scale(domain=(0,100)),

    # tell Altair which field to use to use as the set of columns to be  represented in each group
    column=alt.Column('Name:N', title=None, sort=['C2','C1','C3','LLVM', 'T1','T2','T3'],
                     header=alt.Header(titleFontSize=200, titleOrient='bottom', labelOrient='bottom',labelFontSize=15)),
    order=alt.Order(
      # Sort the segments of the bars by this field
      'val',
      sort='descending'
    ),
    # tell Altair which field to use for color segmentation 
    color=alt.Color('Component:N',
            scale=alt.Scale(
                # make it look pretty with an enjoyable color pallet
                range=['#3d85c6', '#FF7F50','#b43709',"#d74814"],
            ),
        ))\
    .configure_view(
        # remove grid lines around column clusters
        strokeOpacity=0    
    ).configure_axis(
    labelFontSize=15,
    titleFontSize=15
).configure_legend(
     titleFontSize=15,
    labelFontSize=15,
            fillColor='white',
    orient = 'none', 
    legendX = 10,
    legendY = 40,
).properties(
    width=100,
    height=300
)
# chart.save("app_comp.pdf")

alt.Chart(...)